# 01 — Build stage

Join SPADL actions (`event_df`) to running match state derived from the raw
events (`statsbomb.csv`): cumulative goals / yellows / reds per side, each
evaluated *before* the action. Writes `data/stage.csv`.

In [14]:
import warnings
from statsbombpy.api_client import NoAuthWarning
from socceraction.data.statsbomb import StatsBombLoader
import pandas as pd
from statsbombpy import sb
import numpy as np
import socceraction.spadl as spadl

warnings.filterwarnings("ignore", category=NoAuthWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

pd.set_option("display.max_columns", None)  # Show all columns

In [15]:
df = pd.read_csv('data/statsbomb.csv')

  

/var/folders/tv/l4j50kv10gzcv4kzmhysn_fr0000gn/T/ipykernel_3991/2147849192.py:1: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,49,50,54,55,56,57,59,60,61,62,63,64,65,66,67,69,70,71,72,73,75,76,77,78,79,80,85,92,93,94,95,96,97,98,99,100,101,102,103,104,105,107,108,109,111,113,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/statsbomb.csv')


In [16]:
df.head()

,Unnamed: 0,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,counterpress,dribble_no_touch,dribble_nutmeg,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_committed_offensive,foul_committed_penalty,foul_committed_type,foul_won_advantage,foul_won_defensive,foul_won_penalty,goalkeeper_body_part,goalkeeper_end_location,goalkeeper_lost_in_play,goalkeeper_lost_out,goalkeeper_outcome,goalkeeper_penalty_saved_to_post,goalkeeper_position,goalkeeper_punched_out,goalkeeper_saved_to_post,goalkeeper_shot_saved_off_target,goalkeeper_shot_saved_to_post,goalkeeper_success_in_play,goalkeeper_technique,goalkeeper_type,half_start_late_video_start,id,index,injury_stoppage_in_chain,interception_outcome,location,match_id,minute,miscontrol_aerial_won,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_straight,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,player_off_permanent,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_deflected,shot_end_location,shot_first_time,shot_follows_dribble,shot_freeze_frame,shot_key_pass_id,shot_one_on_one,shot_open_goal,shot_outcome,shot_redirect,shot_saved_off_target,shot_saved_to_post,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9d86a178-3514-45d1-9d14-1372e846d17b,1,NaN,NaN,NaN,3754217,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,NaN,1,Chelsea,33,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 4231, 'lineup': [{'player': {'id...",Chelsea,33,00:00:00.000,Starting XI,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82acc213-90f3-4fee-8305-bd9e403cec42,2,NaN,NaN,NaN,3754217,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,NaN,1,Chelsea,33,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 4231, 'lineup': [{'player': {'id...",Arsenal,1,00:00:00.000,Starting XI,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28c7317c-cc33-4181-b64d-b0b07500dcf3,1,NaN,NaN,NaN,3754117,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,NaN,1,Aston Villa,59,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 433, 'lineup': [{'player': {'id'...",Aston Villa,59,00:00:00.000,Starting XI,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85d5fbe4-775a-4ac0-b2ed-23de23977d47,2,NaN,NaN,NaN,3754117,0,NaN,NaN,NaN,NaN,

In [17]:
len(df['match_id'].unique())

matches = list(df["match_id"].unique())

In [20]:
event_df = pd.read_csv('data/event_df.csv')

In [21]:
match_df = sb.matches(competition_id=2, season_id=27)[
    ["match_id", "home_team_id", "home_team", "away_team_id", "away_team"]
]
match_df

,match_id,home_team_id,home_team,away_team_id,away_team
0,3754217,33,Chelsea,1,Arsenal
1,3754117,59,Aston Villa,1,Arsenal
2,3754296,1,Arsenal,36,Manchester City
3,3753983,26,Swansea City,1,Arsenal
4,3754160,1,Arsenal,41,Sunderland
...,...,...,...,...,...
375,3754020,24,Liverpool,28,AFC Bournemouth
376,3754267,23,Watford,27,West Bromwich Albion
377,3754141,1,Arsenal,40,West Ham United
378,3754128,28,AFC Bournemouth,59,Aston Villa


In [22]:
def build_dataset(sb_df, event_df, match_df):

    cumul_df = pd.merge(
        sb_df,
        match_df[
            ["match_id", "home_team_id", "home_team", "away_team_id", "away_team"]
        ],
        how="left",
        left_on="match_id",
        right_on="match_id",
        validate="m:1",
    ).sort_values(['match_id', 'index'])

    is_home = cumul_df["home_team"] == cumul_df["team"]
    cumul_df['is_home'] = is_home

    event_types = {
        "yellow": (cumul_df["foul_committed_card"] == "Yellow Card")
        | (cumul_df.get("bad_behaviour_card") == "Yellow Card"),
        "red": (cumul_df["foul_committed_card"] == "Red Card")
        | (cumul_df.get("bad_behaviour_card") == "Red Card"),
        "second_yellow": (cumul_df["foul_committed_card"] == "Second Yellow")
        | (cumul_df.get("bad_behaviour_card") == "Second Yellow"),
        "goal": (cumul_df["shot_outcome"] == "Goal") | (cumul_df["type"] == "Own Goal For"),
    }

    for event_type, condition in event_types.items():
        cumul_df[f"is_{event_type}"] = condition
        cumul_df[f"is_home_{event_type}"] = is_home & condition
        cumul_df[f"is_away_{event_type}"] = ~is_home & condition

        cumul_df[f"home_{event_type}_count"] = (
            cumul_df.groupby("match_id")[f"is_home_{event_type}"]
            .cumsum()
            .shift(1, fill_value=0)
        )
        cumul_df[f"away_{event_type}_count"] = (
            cumul_df.groupby("match_id")[f"is_away_{event_type}"]
            .cumsum()
            .shift(1, fill_value=0)
        )

    # a second yellow sends the player off: the team gains a red and loses its first yellow
    cumul_df["home_red_count"] += cumul_df["home_second_yellow_count"]
    cumul_df["away_red_count"] += cumul_df["away_second_yellow_count"]
    cumul_df["home_yellow_count"] -= cumul_df["home_second_yellow_count"]
    cumul_df["away_yellow_count"] -= cumul_df["away_second_yellow_count"]

    final = pd.merge(
        event_df,
        cumul_df,
        how="inner",
        left_on=["original_event_id"],
        right_on=["id"],
    )
    
    final['duration'] = final['duration'].clip(lower=0)
    final[['start_x', 'start_y']]

    return final[
        [
            "game_id",
            "original_event_id",
            'index',
            "period_id",
            "minute",
            "second",
            'duration',
            "start_x",
            "start_y",
            "type_name",
            "result_name",
            "is_home",
            "home_yellow_count",
            "away_yellow_count",
            "home_red_count",
            "away_red_count",
            "home_goal_count",
            "away_goal_count",
        ]
    ]

In [23]:
stage = build_dataset(df, event_df, match_df)


In [24]:
stage.head()

,game_id,original_event_id,index,period_id,minute,second,duration,start_x,start_y,type_name,result_name,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count
0,3754217,5f9bdb6d-0379-4f42-9f53-29777f166db5,5,1,0,0,0.386674,51.66875,34.0425,pass,success,False,0,0,0,0,0,0
1,3754217,e5d7db8c-d134-4392-a109-9f6efc7f7225,7,1,0,1,1.499254,53.15625,35.7425,pass,success,False,0,0,0,0,0,0
2,3754217,7ec3ada2-d200-4f0b-ab86-f5a464dc4aa7,9,1,0,2,2.672921,66.80625,32.4275,dribble,success,False,0,0,0,0,0,0
3,3754217,fd5039d0-8dd4-417c-9eac-decd2d267e93,10,1,0,5,0.986454,61.73125,35.4025,pass,success,False,0,0,0,0,0,0
4,3754217,d8a6875b-79c2-4f3b-bb08-38cda23763ec,12,1,0,6,0.851193,49.83125,41.5225,dribble,success,False,0,0,0,0,0,0


In [25]:
stage.describe()

,game_id,index,period_id,minute,second,duration,start_x,start_y,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count
count,7.405100e+05,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000,740510.000000
mean,3.754162e+06,1738.533977,1.492173,44.919063,29.306759,1.485210,53.665043,34.089519,0.508624,0.615595,0.024217,0.026055,0.680289,0.535316
std,1.098090e+02,1016.661927,0.499939,27.023392,17.349147,1.401622,24.949043,20.157829,0.823697,0.898811,0.165181,0.162572,0.971475,0.836217
min,3.753972e+06,5.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.754066e+06,864.000000,1.000000,21.000000,14.000000,0.685754,35.656250,16.617500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.754161e+06,1724.000000,1.000000,45.000000,29.000000,1.230319,54.381250,34.042500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,3.754257e+06,2590.000000,2.000000,68.000000,44.000000,1.950168,72.318750,51.637500,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000
max,3.754351e+06,4261.000000,2.000000,101.000000,59.000000,68.176704,105.000000,68.000000,6.000000,9.000000,2.000000,2.000000,6.000000,6.000000


In [26]:
pd.crosstab(stage["type_name"], stage["result_name"])

result_name,fail,offside,owngoal,red_card,success,yellow_card
type_name,,,,,,
bad_touch,10786,0,38,0,0,0
clearance,0,0,0,0,21645,0
corner_crossed,2177,0,0,0,1080,0
corner_short,89,0,0,0,684,0
cross,6147,67,0,0,2443,0
dribble,0,0,0,0,276949,0
foul,8443,0,0,29,0,1040
freekick_crossed,3057,63,0,0,1755,0
freekick_short,84,4,0,0,4071,0


In [27]:
stage.to_csv('data/stage.csv')